# 1. Setup

**Was dieses Notebook macht:** bevor man eine echte Entry/Exit-Regel für die in `02_tda_clustering` gefundenen Cliquen baut, prüft dieses Notebook schnell und billig, ob ein Kandidaten-Feature (z.B. ein Z-Score, der misst wie weit ein Asset gerade von seiner Clique abweicht) überhaupt mit dem zukünftigen Return zusammenhängt. Eine Korrelationskurve statt einer vollen Backtest-Infrastruktur pro Idee -- siehe Abschnitt 9 für eine vollständige Anleitung, wie man eigene Feature-Kandidaten ergänzt und die Ergebnisse liest.

In [1]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

import src.signals.features as sf
from src.config import (
    ASSET_DAILY_PATH,
    ASSET_ID_COL,
    ASSETS_PATH,
    CLOSE_COL,
    DATE_COL,
    MARKET_SYMBOL_COL,
    MIN_VOLUME_24H_USD,
    OHLCV_DIR,
    PROCESSED_DIR,
    PROJECT_ROOT,
    TIMESTAMP_COL,
    VOLUME_COL,
)
from src.clustering.helpers import HelperFunctions, STABLECOIN_SYMBOLS

helper = HelperFunctions()

print("Imports OK")

Imports OK


# 2. Daten laden & Kalibrierungs-/Backtest-Split

In [2]:
# --- Daten laden & vorverarbeiten (identisch zu 02_tda_clustering) ---
assets_df = pd.read_parquet(ASSETS_PATH)[["asset_id", "symbol"]].drop_duplicates()
id_to_symbol = dict(zip(assets_df["asset_id"], assets_df["symbol"]))
sym_to_id = dict(zip(assets_df["symbol"], assets_df["asset_id"]))

excluded_ids_stablecoins: set[int] = set(
    assets_df.loc[assets_df["symbol"].isin(STABLECOIN_SYMBOLS), "asset_id"].tolist()
)

ohlcv_files = sorted(OHLCV_DIR.glob("*.parquet"))
raw = pd.concat([pd.read_parquet(f) for f in ohlcv_files], ignore_index=True)
raw["base_symbol"] = raw[MARKET_SYMBOL_COL].str.replace("USDT$", "", regex=True)
raw = raw.merge(assets_df.rename(columns={"symbol": "base_symbol"}), on="base_symbol", how="left")
raw = raw.dropna(subset=["asset_id"])
raw["asset_id"] = raw["asset_id"].astype(int)
raw[TIMESTAMP_COL] = pd.to_datetime(raw[TIMESTAMP_COL], utc=True)
raw[DATE_COL] = raw[TIMESTAMP_COL].dt.normalize()
ohlcv_df = raw[[TIMESTAMP_COL, DATE_COL, ASSET_ID_COL, CLOSE_COL]].copy()

data_min = ohlcv_df[DATE_COL].min().date()
data_max = ohlcv_df[DATE_COL].max().date()

universe_df = pd.read_parquet(ASSET_DAILY_PATH)
universe_df[DATE_COL] = pd.to_datetime(universe_df[DATE_COL], utc=True)
all_asset_ids_local = set(universe_df[ASSET_ID_COL].unique())

print(f"Zeitraum: {data_min} \u2192 {data_max}")
print(f"OHLCV: {ohlcv_df.shape[0]:,} Zeilen | Assets: {ohlcv_df[ASSET_ID_COL].nunique()}")

Zeitraum: 2024-07-23 → 2026-07-22
OHLCV: 8,798,178 Zeilen | Assets: 47


In [3]:
# --- Kalibrierungs-/Backtest-Split (identisch zu 02_tda_clustering) ---
# Das Screening läuft NUR auf dem Kalibrierungszeitraum, der Backtest-
# Zeitraum bleibt unangetastet, damit spätere Backtest-Resultate nicht durch
# dieses Screening kontaminiert sind. data_min/data_max/CALIBRATION_END_DATE/
# BACKTEST_START_DATE sind alle reine datetime.date-Objekte (kein .date()
# nochmal draufrufen).
#
# Basket-Zugehörigkeit wird hier NICHT mehr berechnet (kein
# rolling_volume_df/compute_rolling_volume_table nötig) -- die steckt schon
# in den gecachten Cliquen aus 02_tda_clustering/04_clique_cache.ipynb
# (Spalte "basket"), siehe Abschnitt 5.
BACKTEST_FRACTION = 0.2
full_range_days = (data_max - data_min).days + 1
backtest_days = int(full_range_days * BACKTEST_FRACTION)
BACKTEST_START_DATE = data_max - pd.Timedelta(days=backtest_days - 1)
CALIBRATION_END_DATE = BACKTEST_START_DATE - pd.Timedelta(days=1)

print(f"Kalibrierungszeitraum: {data_min} bis {CALIBRATION_END_DATE}")
print(f"Backtest-Zeitraum (unangetastet): {BACKTEST_START_DATE} bis {data_max}")

Kalibrierungszeitraum: 2024-07-23 bis 2026-02-26
Backtest-Zeitraum (unangetastet): 2026-02-27 bis 2026-07-22


# 3. Kalibrierte Parameter (aus 02_tda_clustering übernommen)

Direkt aus 02_pointclouds.ipynb (roh) und 03_residual_pointclouds.ipynb (residualisiert) Abschnitt 4 kopiert, als Literale -- gleiches Muster wie die Widget-Defaults dort, da dieses Notebook seine eigene Kalibrierung nicht wiederholt. Falls dort neu kalibriert wird, müssen diese Werte hier UND in `02_tda_clustering/04_clique_cache.ipynb` von Hand nachgezogen werden (dort neu laufen lassen, um den Cache zu aktualisieren).

**Zwei Zeitschichten, wichtig für den Rest des Notebooks:** die Clique-DISCOVERY (welche Assets gehören zusammen) ist NICHT mehr Teil dieses Notebooks -- sie läuft einmalig in `02_tda_clustering/04_clique_cache.ipynb` und wird hier nur noch als Cache eingelesen (Abschnitt 5). Die SIGNAL-Ebene (Feature + Forward-Return) darunter läuft weiterhin auf 5-Min-Kerzen-Basis, siehe Abschnitt 4. Die Parameter hier werden nur noch für eine Konsistenzprüfung gegen die Cache-Metadaten gebraucht (Abschnitt 5), nicht mehr für eine eigene Berechnung.

In [4]:
# --- Roh (02_pointclouds.ipynb, Abschnitt 4) ---
# Lookback/Threshold neu kalibriert (per korrigierter, nicht-überlappender
# Jaccard-Stabilität statt der alten Tag-vs-Tag-Messung, die durch
# Datenüberlappung lange Lookbacks mechanisch bevorteilt hat -- siehe dortige
# Begründung). Zusätzlich läuft die Clique-Discovery jetzt pro Basket mit
# unterschiedlicher Kadenz (BEST_CADENCE_HOURS_RAW, aus der Cache-Meta
# geprüft, siehe Abschnitt 5) statt einheitlich täglich -- Basket 1 bleibt
# bei 24h, Basket 2-4 laufen mit 12h.
BEST_LOOKBACK_RAW: dict[str, int] = {
    "Basket 1": 7,
    "Basket 2": 10,
    "Basket 3": 15,
    "Basket 4": 5,
}
BEST_THRESHOLD_RAW: dict[str, float] = {
    "Basket 1": 0.1916,
    "Basket 2": 0.1085,
    "Basket 3": 0.1272,
    "Basket 4": 0.2570,
}
BEST_CADENCE_HOURS_RAW: dict[str, int] = {
    "Basket 1": 24,
    "Basket 2": 12,
    "Basket 3": 12,
    "Basket 4": 12,
}
EPSILON_CAP_RAW = 1.0

# --- Residualisiert (03_residual_pointclouds.ipynb, Abschnitt 4) ---
# Unverändert -- residualisierte Variante wurde in diesem Durchgang nicht neu
# kalibriert (weiterhin tägliche Discovery).
FACTOR_COLUMNS_RESIDUAL = ["BTC", "ETH"]
REGRESSION_LOOKBACK_DAYS = 60
BEST_LOOKBACK_RESIDUAL: dict[str, int] = {
    "Basket 1": 10,
    "Basket 2": 30,
    "Basket 3": 35,
    "Basket 4": 40,
}
BEST_THRESHOLD_RESIDUAL: dict[str, float] = {
    "Basket 1": 0.1599,
    "Basket 2": 0.0849,
    "Basket 3": 0.1025,
    "Basket 4": 0.1783,
}
EPSILON_CAP_RESIDUAL = 1.3

factor_asset_ids: set[int] = {sym_to_id[s] for s in FACTOR_COLUMNS_RESIDUAL if s in sym_to_id}

print("Roh:            Lookback", BEST_LOOKBACK_RAW, "| Kadenz(h)", BEST_CADENCE_HOURS_RAW, "| epsilon_cap", EPSILON_CAP_RAW)
print("Residualisiert: Lookback", BEST_LOOKBACK_RESIDUAL, "| epsilon_cap", EPSILON_CAP_RESIDUAL)

Roh:            Lookback {'Basket 1': 7, 'Basket 2': 10, 'Basket 3': 15, 'Basket 4': 5} | Kadenz(h) {'Basket 1': 24, 'Basket 2': 12, 'Basket 3': 12, 'Basket 4': 12} | epsilon_cap 1.0
Residualisiert: Lookback {'Basket 1': 10, 'Basket 2': 30, 'Basket 3': 35, 'Basket 4': 40} | epsilon_cap 1.3


# 4. Kerzen-Returns, Vola, Horizonte

**Entry-Ebene (5-Min-Kerzen statt Kalendertage):** `returns_df` ist die volle 5-Min-Kerzen-Returnmatrix (kein Aggregieren mehr auf Tagesebene). Innerhalb jedes Discovery-Tages (aus dem Cache, siehe Abschnitt 5) wird nicht nur EIN Feature-Wert berechnet, sondern einer PRO Kerze (mit `ENTRY_STRIDE_CANDLES` einstellbar, wie dicht) -- so bekommt man ein saubereres Bild als mit nur einem Tageswert.

**Horizont-Ebene (dicht statt wenige Stützpunkte):** `HORIZONS` deckt JEDE Kerze bis zu `MAX_HORIZON_DAYS` ab (nicht nur ein paar ausgewählte Punkte wie 1/2/3/5/10 Tage) -- das ergibt eine glatte Korrelationskurve statt einer groben Tabelle. Forward-Fenster dürfen über Mitternacht hinausreichen (kein Zwang, den Trade am selben Tag zu schliessen), die Clique-MITGLIEDSCHAFT bleibt aber für den ganzen Discovery-Tag fix.

**Achtung Speicher/Laufzeit:** dichte Entries x dichte Horizonte multipliziert sich schnell. Die Panel-Sammlung selbst ist vektorisiert (kein Python-Loop pro Horizont), aber die Zeilenzahl (Entries x Assets pro Clique) wächst linear mit `ENTRY_STRIDE_CANDLES` kleiner / `DISCOVERY_DAY_STRIDE` kleiner. Bei Bedarf hier zuerst hochsetzen, um einen schnellen Testlauf zu bekommen, bevor man die volle Auflösung laufen lässt.

In [5]:
CANDLES_PER_DAY = 24 * 60 // 5  # 288, siehe config.OHLCV_DIR = asset_ohlcv_5m

returns_df = sf.build_returns_matrix(
    ohlcv_df, timestamp_column=TIMESTAMP_COL, asset_id_column=ASSET_ID_COL, close_column=CLOSE_COL,
)
trailing_vol_df = sf.build_trailing_vol(returns_df, window_candles=CANDLES_PER_DAY)

MAX_HORIZON_DAYS = 2  # Forward-Fenster darf über Mitternacht hinaus, hier bis zu 2 Tage
HORIZONS = np.arange(1, MAX_HORIZON_DAYS * CANDLES_PER_DAY + 1)  # dicht: jede Kerze
HORIZON_COLS = [f"fwd_ret_{h}c" for h in HORIZONS]

ENTRY_STRIDE_CANDLES = 6   # jede 6. Kerze (=alle 30 Min) als Entry-Kandidat
DISCOVERY_DAY_STRIDE = 3   # jeder 3. Kalendertag als Discovery-Tag (Filter auf den Cache, siehe Abschnitt 5)

REGRESSION_LOOKBACK_CANDLES = REGRESSION_LOOKBACK_DAYS * CANDLES_PER_DAY

# first_valid: der spätere von (a) rohem Lookback-Bedarf, (b) residualisiertem
# Regressionsfenster-Bedarf -- damit dieselbe Discovery-Tag-Liste für beide
# Varianten fair vergleichbar bleibt.
first_valid = data_min + pd.Timedelta(
    days=max(REGRESSION_LOOKBACK_DAYS, max(BEST_LOOKBACK_RAW.values()), max(BEST_LOOKBACK_RESIDUAL.values())) - 1
)
# last_valid: Puffer von MAX_HORIZON_DAYS + 1, damit kein Forward-Fenster in
# den (unangetasteten) Backtest-Zeitraum hineinreicht.
last_valid = CALIBRATION_END_DATE - pd.Timedelta(days=MAX_HORIZON_DAYS + 1)

# Ziel-Discovery-Tage: nur noch ein FILTER auf den Cliquen-Cache (Abschnitt 5),
# treibt selbst keine Berechnung mehr an.
discovery_dates = [
    d.date() for d in pd.date_range(data_min, data_max, freq="D", tz="UTC")
    if first_valid <= d.date() <= last_valid
][::DISCOVERY_DAY_STRIDE]
discovery_dates_set = set(discovery_dates)

print(f"{len(discovery_dates)} Ziel-Discovery-Tage ({discovery_dates[0]} bis {discovery_dates[-1]})")
print(f"{len(HORIZONS)} Horizonte (1 bis {HORIZONS[-1]} Kerzen = {HORIZONS[-1] * 5 / 60:.1f}h)")
print(f"Entry-Stride: {ENTRY_STRIDE_CANDLES} Kerzen (alle {ENTRY_STRIDE_CANDLES * 5} Min)")

174 Ziel-Discovery-Tage (2024-09-20 bis 2026-02-21)
576 Horizonte (1 bis 576 Kerzen = 48.0h)
Entry-Stride: 6 Kerzen (alle 30 Min)


# 5. Cliquen-Cache einlesen

Liest die in `02_tda_clustering/04_clique_cache.ipynb` vorberechneten Cliquen ein (`data/processed/cliques/`) statt die teure Discovery hier zu wiederholen. Prüft dabei, ob die dort verwendeten Kalibrierungsparameter noch mit den in Abschnitt 3 hinterlegten übereinstimmen (Cache-Staleness-Check) -- bei einer Warnung `04_clique_cache.ipynb` neu laufen lassen.

Der Cache selbst deckt den GESAMTEN verfügbaren Zeitraum ab (auch den Backtest-Zeitraum, für spätere Wiederverwendung z.B. in der Backtest-Engine) -- hier wird auf `discovery_dates_set` (Kalibrierungszeitraum + Stride, siehe Abschnitt 4) gefiltert, damit dieses Notebook den Backtest-Zeitraum weiterhin nicht anfasst.

In [6]:
CLIQUES_DIR = PROCESSED_DIR / "cliques"


def _load_cliques_cache(variant: str, expected_meta: dict) -> pd.DataFrame:
    path = CLIQUES_DIR / f"cliques_{variant}.parquet"
    meta_path = CLIQUES_DIR / f"cliques_{variant}.meta.json"
    if not path.exists():
        raise FileNotFoundError(
            f"{path.relative_to(PROJECT_ROOT)} nicht gefunden -- zuerst "
            "02_tda_clustering/04_clique_cache.ipynb laufen lassen."
        )
    df = pd.read_parquet(path)
    df["discovery_date"] = pd.to_datetime(df["discovery_date"], utc=True)
    df["asset_ids"] = df["asset_ids"].apply(list)  # Parquet gibt Listen als np.ndarray zurück

    meta = json.load(open(meta_path))
    mismatches = [k for k in expected_meta if meta.get(k) != expected_meta[k]]
    if mismatches:
        print(
            f"WARNUNG: Cache '{variant}' wurde mit anderen Parametern berechnet als hier "
            f"hinterlegt ({mismatches}) -- 04_clique_cache.ipynb neu laufen lassen, sonst "
            "sind Discovery und Feature-Screening inkonsistent kalibriert."
        )
    return df


def _filter_to_target_dates(df: pd.DataFrame) -> pd.DataFrame:
    return df[df["discovery_date"].dt.date.isin(discovery_dates_set)].reset_index(drop=True)


cliques_raw = _filter_to_target_dates(_load_cliques_cache("raw", {
    "best_lookback": BEST_LOOKBACK_RAW, "best_threshold": BEST_THRESHOLD_RAW, "epsilon_cap": EPSILON_CAP_RAW,
    "best_cadence_hours": BEST_CADENCE_HOURS_RAW,
}))
cliques_residual = _filter_to_target_dates(_load_cliques_cache("residual", {
    "best_lookback": BEST_LOOKBACK_RESIDUAL, "best_threshold": BEST_THRESHOLD_RESIDUAL,
    "epsilon_cap": EPSILON_CAP_RESIDUAL, "factor_columns": FACTOR_COLUMNS_RESIDUAL,
    "regression_lookback_days": REGRESSION_LOOKBACK_DAYS,
}))

print(f"cliques_raw: {len(cliques_raw)} Cliquen über {cliques_raw['discovery_date'].nunique()} Tage")
print(f"cliques_residual: {len(cliques_residual)} Cliquen über {cliques_residual['discovery_date'].nunique()} Tage")

cliques_raw: 792 Cliquen über 330 Tage
cliques_residual: 324 Cliquen über 147 Tage


# 6. Panel sammeln: roh

Nutzt die aus dem Cache geladene `cliques_raw` (Abschnitt 5) -- keine erneute Clique-Discovery mehr nötig, nur noch die schnelle, vektorisierte Kerzen-/Feature-Berechnung.

In [7]:
RUN_SCREENING = True  # auf False setzen, sobald ein Ergebnis vorliegt, das man nicht neu rechnen will

if RUN_SCREENING:
    panel_raw = sf.collect_screening_panel(
        cliques_df=cliques_raw, id_to_symbol=id_to_symbol,
        returns_df=returns_df, trailing_vol_df=trailing_vol_df,
        entry_stride_candles=ENTRY_STRIDE_CANDLES,
        horizons=HORIZONS, feature_fns=sf.FEATURE_REGISTRY, residualize=False,
    )
    print(f"panel_raw: {len(panel_raw):,} Zeilen, {panel_raw['discovery_date'].nunique() if len(panel_raw) else 0} Discovery-Tage")
else:
    print("RUN_SCREENING=False -- panel_raw nicht neu berechnet.")

panel_raw: 331,632 Zeilen, 166 Discovery-Tage


# 7. Panel sammeln: residualisiert

Nutzt die aus dem Cache geladene `cliques_residual` (Abschnitt 5). Die Residualisierung der Signal-Ebene selbst (`factor_asset_ids`, kerzenbasiert) passiert INNERHALB von `collect_screening_panel` (eingefrorenes Beta pro Discovery-Tag, siehe `features.compute_frozen_betas`) -- unabhängig von der Residualisierung, die `04_clique_cache.ipynb` für die Clique-DISCOVERY selbst gemacht hat.

In [8]:
if RUN_SCREENING:
    panel_residual = sf.collect_screening_panel(
        cliques_df=cliques_residual, id_to_symbol=id_to_symbol,
        returns_df=returns_df, trailing_vol_df=trailing_vol_df,
        entry_stride_candles=ENTRY_STRIDE_CANDLES,
        horizons=HORIZONS, feature_fns=sf.FEATURE_REGISTRY,
        residualize=True, factor_asset_ids=list(factor_asset_ids),
        regression_lookback_candles=REGRESSION_LOOKBACK_CANDLES,
    )
    print(f"panel_residual: {len(panel_residual):,} Zeilen, {panel_residual['discovery_date'].nunique() if len(panel_residual) else 0} Discovery-Tage")
else:
    print("RUN_SCREENING=False -- panel_residual nicht neu berechnet.")

panel_residual: 121,920 Zeilen, 147 Discovery-Tage


# 8. Feature-Vergleich: Korrelationskurve über den Horizont

Negative Werte = Evidenz für Mean-Reversion (hoher Feature-Wert jetzt -> negativer Forward-Return relativ zur Clique). Bewusst nur die reine Kurve, ohne Konfidenzband -- Form und Niveau werden direkt im Plot beurteilt (fällt sie schnell auf 0 ab, oder bleibt sie über viele Horizonte stabil negativ?).

`FEATURE_COLS` hier zentral definiert -- neue Feature-Kandidaten einfach in `features.FEATURE_REGISTRY` ergänzen (siehe Abschnitt 9), sie tauchen dann automatisch in dieser und allen folgenden Zellen auf.

In [9]:
FEATURE_COLS = list(sf.FEATURE_REGISTRY.keys())

curves_raw = {feat: sf.correlation_curve(panel_raw, feat, HORIZON_COLS) for feat in FEATURE_COLS}
curves_residual = {feat: sf.correlation_curve(panel_residual, feat, HORIZON_COLS) for feat in FEATURE_COLS}

# Interaktiv (Plotly): eine Figur, klickbare Legende zum Ein-/Ausblenden
# einzelner Linien, Hover zeigt Korrelation + n_obs pro Horizont.
sf.plot_correlation_curves_interactive({"roh": curves_raw, "residualisiert": curves_residual})

# 9. Wie geht es weiter: eigene Feature-Kandidaten ergänzen

Dieser Abschnitt ist die Anleitung, um selbstständig neue Signal-Ideen zu testen, ohne den Rest des Notebooks verstehen/anfassen zu müssen.

## Was das Notebook tut (kurz)

1. **Clique-Discovery**: NICHT Teil dieses Notebooks -- läuft einmalig in `02_tda_clustering/04_clique_cache.ipynb` (teuer: volle Persistenzberechnung pro Tag/Basket) und wird hier nur als Cache eingelesen (Abschnitt 5).
2. **Panel-Sammlung**: für jeden Discovery-Tag werden ALLE 5-Min-Kerzen dieses Tages (im Abstand `ENTRY_STRIDE_CANDLES`) als "Entry-Kandidaten" durchgegangen. Pro Entry UND pro Asset in der Clique wird eine Zeile geschrieben: die Feature-Werte JETZT, plus der (Clique-relative) Forward-Return für JEDEN Horizont bis `MAX_HORIZON_DAYS`.
3. **Vergleich**: Korrelation zwischen Feature-Wert und Forward-Return, als Kurve über den Horizont (Abschnitt 8).

## Eigenes Feature hinzufügen (in `src/signals/features.py`)

1. Neue Funktion schreiben, Signatur `(ctx: CliqueContext) -> pd.Series` (Index=Symbol, ein Wert pro Asset in der Clique). Verfügbare Felder in `ctx`: `candle_return` (Kerzen-Return relativ zur Clique-Kohorte), `trailing_vol` (rollierende Vola), `persistence` (wie robust die Clique ist), `lookback_days`, `timestamp`, `discovery_date`, `basket`.
   ```python
   def feature_mein_kandidat(ctx: CliqueContext) -> pd.Series:
       # z.B. Momentum statt Reversion, oder eine andere Normalisierung
       ...
       return pd.Series(werte, index=ctx.asset_syms)
   ```
2. In `FEATURE_REGISTRY` eintragen: `FEATURE_REGISTRY["mein_kandidat"] = feature_mein_kandidat`.
3. Notebook neu laufen lassen (oder nur ab Abschnitt 6, falls `returns_df`/`cliques_raw`/`cliques_residual` schon stehen) -- das neue Feature taucht automatisch in der Kurve (Abschnitt 8) auf, ohne dass irgendeine andere Zelle angefasst werden muss.

**Braucht das Feature mehr Kontext als `CliqueContext` aktuell hat** (z.B. ein längeres Momentum-Fenster, Orderbuch-Daten, etc.)? `CliqueContext` in `src/signals/features.py` um ein neues Feld erweitern und in `collect_screening_panel` befüllen -- bestehende Features lesen weiterhin nur die Felder, die sie kennen, nichts bricht.

## Wie man die Ergebnisse liest

- **Vorzeichen:** negativ = Mean-Reversion (das ist die Hypothese dieser ganzen Arbeit). Positiv wäre eher ein Momentum-Signal -- nicht per se falsch, aber eine andere Handelsidee als die Clique-Reversion-These.
- **Form der Kurve:** fällt die Korrelation von Horizont 1 auf 0 sehr schnell ab -> das Signal ist nur für extrem kurze Haltedauern brauchbar (ggf. Mikrostruktur-Rauschen, nicht robust genug für echte Order-Latenz). Ein Signal, das über mehrere Stunden robust negativ bleibt, ist der interessantere Kandidat. Der Plot zeigt bewusst nur die reine Kurve (kein Konfidenzband) -- Robustheit wird direkt visuell beurteilt: bleibt die Kurve über viele Horizonte stabil, oder ist es ein einzelner Ausreisser?
- **Roh vs. residualisiert:** falls die residualisierte Kurve durchgehend stärker negativ ist als die rohe, bestätigt das den Sinn der ganzen Residualisierungs-Übung aus `03_residual_pointclouds.ipynb` auch für den Trade selbst, nicht nur für sauberere Cluster.

## Nächster Schritt nach einem guten Kandidaten

Ein Feature mit robust negativer, über mehrere Horizonte stabiler Korrelation (Abschnitt 8) ist reif für ein eigenes Signal-Notebook (`02_<name>_signal.ipynb`) mit echten Entry/Exit-Regeln und Position Sizing -- dieses Screening-Notebook selbst bleibt bewusst ohne Handelslogik, es beantwortet nur "lohnt es sich überhaupt, dafür eine zu bauen".

## Konkret geplant: paarweise Spreads (Lin et al., Additional-Info-PDF Abschnitt 3)

Der Z-Score hier ist direkt aus der Haupt-PDF (Abschnitt 4.2) und wird dort selbst als
"willkürlicher" Schwellenwert-Ansatz bezeichnet. Die Additional-Info-PDF schlägt als
Ersatz/Ergänzung die Methode aus Lin et al. (2021) vor -- datengetrieben statt willkürlich.
Das ist der nächste Baustein, NICHT mehr Teil dieses Notebooks (andere Panel-Struktur:
pro Clique-PAAR statt pro Asset), aber hier als Fahrplan festgehalten:

1. **Spread pro Paar**: für jedes der (n über 2) Paare in einer Clique die einfache
   Return-Differenz `R_t = r_i(t) - r_j(t)` auf **Log-Return-Basis** (Konvention aus der
   Additional-Info-PDF, dort explizit "daily log-returns"). Kein Hedge-Ratio, kein
   Log-Preis-Spread wie beim klassischen Cointegration-Pairs-Trading -- das PDF verlangt
   nur die simple Return-Differenz.
2. **Vola-Adjustierung**: `x_t = R_t / sigma_hat_t`, `sigma_hat_t` per EWMA (RiskMetrics)
   oder Student-t-GARCH geschätzt.
3. **Tolerance-Interval-Signal**: statt eines fixen Z-Score-Schwellenwerts ein
   nicht-parametrisches, einseitiges Wilks-Tolerance-Limit (p-content=95%,
   confidence=95%) auf die historischen `x_t`-Werte des Paars. Entry, wenn der
   1-Schritt-Forecast die obere/untere Grenze durchbricht; Exit, wenn der Spread wieder
   Richtung Null dreht.

**Explizit NICHT gebraucht** (im Unterschied zum klassischen Pairs-Trading): Cointegration-
/ADF-Tests und Ornstein-Uhlenbeck-Modellierung. Beides adressiert Probleme von
Preis-Level-Spreads; hier wird mit Return-Differenzen gearbeitet, die bereits (durch die
Differenzierung selbst, plus die TDA-Vorfilterung auf co-bewegte Assets) näherungsweise
stationär sind.

**Vor dem vollen Aufbau (EWMA/GARCH + Tolerance-Interval)**: erst `x_t` als neues Feature
durch eine paar-basierte Variante dieses Screening-Panels laufen lassen (gleiches Prinzip
wie oben mit dem Z-Score) -- billige Validierung, ob die Idee überhaupt trägt, bevor die
volle Tolerance-Interval-Maschinerie gebaut wird.